# Delta Lake Demo: Predict → Store → Load → Inspect

This notebook demonstrates the full workflow for AnalysisGNN inference results
stored in Delta Lake format:

1. Run inference with no aggregation
2. Write results to Delta Lake (skipped if already present)
3. Load and inspect notes, edges, probabilities, hyperedges
4. Query top-k predictions and argmax summaries
5. Export to CSV

## Setup

In [1]:
import os
import pandas as pd

# Change to the repo root so all relative paths work
REPO_ROOT = os.path.dirname(os.path.abspath(""))
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

SCORE_PATH = "notebooks/Minuet_in_G_Major_K.1.musicxml"
CHECKPOINT_PATH = "artifacts/gradio_checkpoints/uocj8f6y_full_last.ckpt"
OUTPUT_DIR = "outputs/Minuet_in_G_Major_K.1"

from analysisgnn.storage import delta_reader, delta_writer

Working directory: /home/laser/git/analysisgnn


/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Run Inference (with no aggregation)

Load the model and run inference. The raw per-note softmax outputs are returned
(no onset/beat/measure averaging).

In [2]:
from analysisgnn.models.analysis import ContinualAnalysisGNN

model = ContinualAnalysisGNN.load_from_checkpoint(
    CHECKPOINT_PATH, map_location="cpu", strict=False
)
model.eval()

predictions, intermediates = model.predict(
    SCORE_PATH,
    aggregation_spec={"mode": "none"},
    return_intermediates=True,
)

print(f"Tasks: {list(predictions.keys())}")
for task, tensor in predictions.items():
    if hasattr(tensor, 'shape') and tensor.ndim == 2:
        print(f"  {task}: {tensor.shape[0]} notes x {tensor.shape[1]} classes")

/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/partitura/directions.py:443: UserWarning: unhandled: Fine
  warnings.warn("unhandled: {}".format(string[start:end]))
/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/partitura/directions.py:524: UserWarning: error parsing "Trio" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/partitura/directions.py:524: UserWarning: error parsing "Minuetto da Capo al Fine" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
/home/laser/miniconda3/envs/analysisgnn/lib/python3.11/site-packages/partitura/io/importmusicxml.py:462: UserWarning: Found repeat without start
Starting point 0 is assumed
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at manoskary/musicbert-large and are newly initialized: ['pooler.dens

Tasks: ['cadence', 'localkey', 'tonkey', 'quality', 'inversion', 'root', 'bass', 'degree1', 'degree2', 'hrythm', 'pcset', 'romanNumeral', 'section', 'phrase', 'organ_point', 'tpc_in_label', 'tpc_is_root', 'tpc_is_bass', 'downbeat', 'note_degree', 'staff']
  cadence: 256 notes x 4 classes
  localkey: 256 notes x 50 classes
  tonkey: 256 notes x 50 classes
  quality: 256 notes x 15 classes
  inversion: 256 notes x 4 classes
  root: 256 notes x 38 classes
  bass: 256 notes x 38 classes
  degree1: 256 notes x 22 classes
  degree2: 256 notes x 22 classes
  hrythm: 256 notes x 2 classes
  pcset: 256 notes x 94 classes
  romanNumeral: 256 notes x 185 classes
  section: 256 notes x 2 classes
  phrase: 256 notes x 2 classes
  organ_point: 256 notes x 2 classes
  tpc_in_label: 256 notes x 2 classes
  tpc_is_root: 256 notes x 2 classes
  tpc_is_bass: 256 notes x 2 classes
  downbeat: 256 notes x 45 classes
  note_degree: 256 notes x 49 classes
  staff: 256 notes x 4 classes


## Write Delta Lake

Write the inference results to Delta Lake. The write is **skipped** if the output
already exists, to avoid polluting the Delta Log version history with duplicate
snapshots. See AGENTS.md for the rationale.

In [3]:
# Guard: only write if the Delta Lake does not already exist
if not os.path.isdir(os.path.join(OUTPUT_DIR, "notes", "_delta_log")):
    delta_writer.write_analysis_results(
        output_dir=OUTPUT_DIR,
        score=intermediates["score"],
        note_array=intermediates["note_array"],
        predictions=predictions,
        data=intermediates["data"],
        task_dict=model.task_dict,
        metadata={
            "score_path": SCORE_PATH,
            "full_checkpoint": CHECKPOINT_PATH,
            "device": "cpu",
        },
    )
    print(f"Delta Lake written to {OUTPUT_DIR}")
else:
    print(f"Delta Lake already exists at {OUTPUT_DIR} \u2014 skipping write")

Delta Lake already exists at outputs/Minuet_in_G_Major_K.1 — skipping write


## Load and Inspect Notes

In [4]:
notes = delta_reader.load_notes(OUTPUT_DIR)
print(f"{len(notes)} notes, columns: {list(notes.columns)}")
notes.head(10)

256 notes, columns: ['note_id', 'onset_div', 'onset_beat', 'duration_div', 'duration_beat', 'pitch_midi', 'pitch_spelling', 'staff', 'voice', 'measure', 'ts_beats']


,note_id,onset_div,onset_beat,duration_div,duration_beat,pitch_midi,pitch_spelling,staff,voice,measure,ts_beats
0,p0n0,0,-1.0,6,0.5,83,B4,1,1,1,3.0
1,p0n2,6,-0.5,6,0.5,79,G4,1,1,1,3.0
2,p0n4,12,0.0,12,1.0,55,G2,2,5,2,3.0
3,p0n3,12,0.0,12,1.0,71,B3,1,1,2,3.0
4,p0n6,24,1.0,12,1.0,57,A2,2,5,2,3.0
5,p0n5,24,1.0,12,1.0,72,C4,1,1,2,3.0
6,p0n8,36,2.0,12,1.0,59,B2,2,5,2,3.0
7,p0n7,36,2.0,12,1.0,74,D4,1,1,2,3.0
8,p0n10,48,3.0,12,1.0,59,B2,2,5,3,3.0
9,p0n9,48,3.0,12,1.0,74,D4,1,1,3,3.0


## Inspect Edges

In [5]:
edges = delta_reader.load_edges(OUTPUT_DIR)
print(f"{len(edges)} total edges")
edges.groupby("edge_type").size()

1094 total edges


edge_type
consecutive    519
during          47
onset          528
dtype: int64

## Query Top-3 Predictions for `romanNumeral`

In [6]:
top3_rn = delta_reader.load_probabilities(OUTPUT_DIR, task="romanNumeral", top_k=3)
top3_rn = top3_rn.sort_values(["note_id", "rank"])
print(f"{len(top3_rn)} rows (top-3 per note for romanNumeral)")
top3_rn.head(15)  # Shows top-3 candidates for the first 5 notes

768 rows (top-3 per note for romanNumeral)


,note_id,task,class_id,class_label,probability,is_argmax,rank
0,p0n0,romanNumeral,1,I,0.874509,True,1
285,p0n0,romanNumeral,3,V,0.032111,False,2
480,p0n0,romanNumeral,6,IV,0.004789,False,3
6,p0n10,romanNumeral,1,I,0.842220,True,1
291,p0n10,romanNumeral,3,V,0.033301,False,2
741,p0n10,romanNumeral,19,iii,0.015092,False,3
586,p0n100,romanNumeral,7,ii,0.639484,True,1
502,p0n100,romanNumeral,6,IV,0.127909,False,2
712,p0n100,romanNumeral,12,ii7,0.058487,False,3
587,p0n101,romanNumeral,7,ii,0.832145,True,1


## Argmax Summary (Wide Format)

One row per note, with the argmax class label and confidence for selected tasks.

In [7]:
summary = delta_reader.argmax_summary(OUTPUT_DIR, tasks=["romanNumeral", "localkey", "quality", "cadence"])
print(f"{len(summary)} notes, {len(summary.columns)} columns")
summary.head(10)

256 notes, 19 columns


,note_id,onset_div,onset_beat,duration_div,duration_beat,pitch_midi,pitch_spelling,staff,voice,measure,ts_beats,cadence,localkey,quality,romanNumeral,cadence_confidence,localkey_confidence,quality_confidence,romanNumeral_confidence
0,p0n0,0,-1.0,6,0.5,83,B4,1,1,1,3.0,,G,major triad,I,0.926015,0.936767,0.888394,0.874509
1,p0n2,6,-0.5,6,0.5,79,G4,1,1,1,3.0,,G,major triad,I,0.925198,0.939204,0.887082,0.890107
2,p0n4,12,0.0,12,1.0,55,G2,2,5,2,3.0,,G,major triad,I,0.914487,0.938632,0.886803,0.885560
3,p0n3,12,0.0,12,1.0,71,B3,1,1,2,3.0,,G,major triad,I,0.915580,0.938534,0.887698,0.883164
4,p0n6,24,1.0,12,1.0,57,A2,2,5,2,3.0,,G,minor triad,viio,0.916271,0.930004,0.275194,0.225330
5,p0n5,24,1.0,12,1.0,72,C4,1,1,2,3.0,,G,minor triad,viio,0.917480,0.932558,0.261895,0.225389
6,p0n8,36,2.0,12,1.0,59,B2,2,5,2,3.0,,G,major triad,I,0.918999,0.936582,0.895961,0.889391
7,p0n7,36,2.0,12,1.0,74,D4,1,1,2,3.0,,G,major triad,I,0.918279,0.936965,0.897614,0.896599
8,p0n10,48,3.0,12,1.0,59,B2,2,5,3,3.0,,G,major triad,I,0.925429,0.935698,0.867195,0.842220
9,p0n9,48,3.0,12,1.0,74,D4,1,1,3,3.0,,G,major triad,I,0.923129,0.936319,0.868595,0.853815


## Inspect Hyperedge Groupings

In [11]:
print("Group types:", delta_reader.list_group_types(OUTPUT_DIR))

beat_groups = delta_reader.load_hyperedges(OUTPUT_DIR, edge_type="beat")
print(f"\nBeat groups: {beat_groups['group_id'].nunique()} groups, {len(beat_groups)} memberships")
beat_groups.head()

Group types: ['beat', 'measure', 'onset']

Beat groups: 95 groups, 256 memberships


,group_id,note_id,edge_type,parent_group_id
0,beat_0,p0n0,beat,NaN
1,beat_0,p0n2,beat,NaN
2,beat_1,p0n4,beat,NaN
3,beat_1,p0n3,beat,NaN
4,beat_2,p0n6,beat,NaN


## Metadata

In [9]:
meta = delta_reader.load_metadata(OUTPUT_DIR)
print(f"Score: {meta['score_id']}, {meta['num_notes']} notes, {len(meta['task_dict'])} tasks")
print(f"Edge types: {meta['edge_types_included']}")
print(f"Hyperedge types: {meta['hyperedge_types']}")
print(f"Inference timestamp: {meta['inference_timestamp']}")

Score: Minuet_in_G_Major_K.1, 256 notes, 21 tasks
Edge types: ['during', 'onset', 'consecutive']
Hyperedge types: ['onset', 'beat', 'measure']
Inference timestamp: 2026-03-23T10:11:18.880601+00:00


## CSV Export

In [10]:
csv_path = delta_reader.export_table_to_csv(OUTPUT_DIR, "notes", "/tmp/notes.csv")
print(f"Exported to {csv_path}")

pd.read_csv(csv_path).head(5)

Exported to /tmp/notes.csv


,note_id,onset_div,onset_beat,duration_div,duration_beat,pitch_midi,pitch_spelling,staff,voice,measure,ts_beats
0,p0n0,0,-1.0,6,0.5,83,B4,1,1,1,3.0
1,p0n2,6,-0.5,6,0.5,79,G4,1,1,1,3.0
2,p0n4,12,0.0,12,1.0,55,G2,2,5,2,3.0
3,p0n3,12,0.0,12,1.0,71,B3,1,1,2,3.0
4,p0n6,24,1.0,12,1.0,57,A2,2,5,2,3.0
